# Silver Layer — Clean & Enrich Media Data
Deduplicates, parses dates, derives subscriber tenure, churn flags, content completion rates, and ad CTR.

In [ ]:
from pyspark.sql.functions import (
    col, to_date, datediff, round as spark_round, when, lit,
    trim, coalesce, current_timestamp, current_date
)

In [ ]:
# Silver content catalog — clean types, trim strings
df_content = spark.read.format('delta').table('bronze_content_catalog')
df_content = (
    df_content
    .dropDuplicates(['content_id'])
    .withColumn('title',        trim(col('title')))
    .withColumn('genre',        trim(col('genre')))
    .withColumn('content_type', trim(col('content_type')))
    .withColumn('language',     trim(col('language')))
    .withColumn('silver_timestamp', current_timestamp())
)
df_content.write.mode('overwrite').format('delta').saveAsTable('silver_content_catalog')
print(f'Silver content catalog: {df_content.count()} rows')

In [ ]:
# Silver subscribers — parse dates, derive tenure, active flag, plan_tier
df_subs = spark.read.format('delta').table('bronze_subscribers')
df_subs = (
    df_subs
    .dropDuplicates(['subscriber_id'])
    .withColumn('signup_date', to_date(col('signup_date'), 'yyyy-MM-dd'))
    .withColumn('churn_date',
        when(col('churn_date') != '', to_date(col('churn_date'), 'yyyy-MM-dd'))
        .otherwise(lit(None))
    )
    .withColumn('tenure_days',
        when(col('is_churned') == 1,
            datediff(col('churn_date'), col('signup_date'))
        ).otherwise(
            datediff(current_date(), col('signup_date'))
        )
    )
    .withColumn('tenure_bucket',
        when(col('tenure_days') < 90,  '0-3 months')
        .when(col('tenure_days') < 180, '3-6 months')
        .when(col('tenure_days') < 365, '6-12 months')
        .when(col('tenure_days') < 730, '1-2 years')
        .otherwise('2+ years')
    )
    .withColumn('plan_tier',
        when(col('plan_type') == 'Premium', 'High')
        .when(col('plan_type') == 'Standard', 'Mid')
        .otherwise('Low')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
df_subs.write.mode('overwrite').format('delta').saveAsTable('silver_subscribers')
print(f'Silver subscribers: {df_subs.count()} rows')

In [ ]:
# Silver viewing history — parse date, derive completion_rate, engagement_band
df_views = spark.read.format('delta').table('bronze_viewing_history')
df_content_dur = spark.read.format('delta').table('silver_content_catalog').select('content_id', 'duration_mins')

df_views = (
    df_views
    .dropDuplicates(['view_id'])
    .withColumn('view_date', to_date(col('view_date'), 'yyyy-MM-dd'))
    .withColumn('rating', when(col('rating') == '', lit(None)).otherwise(col('rating').cast('int')))
    .join(df_content_dur, 'content_id', 'left')
    .withColumn('completion_pct',
        spark_round(col('watch_duration_mins') / col('duration_mins') * 100, 1)
    )
    .withColumn('engagement_band',
        when(col('completion_pct') >= 85, 'Completed')
        .when(col('completion_pct') >= 50, 'Partial')
        .when(col('completion_pct') >= 20, 'Sampled')
        .otherwise('Abandoned')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
df_views.write.mode('overwrite').format('delta').saveAsTable('silver_viewing_history')
print(f'Silver viewing history: {df_views.count()} rows')

In [ ]:
# Silver ad impressions — parse date, derive CTR band
df_ads = spark.read.format('delta').table('bronze_ad_impressions')
df_ads = (
    df_ads
    .dropDuplicates(['impression_id'])
    .withColumn('ad_date', to_date(col('ad_date'), 'yyyy-MM-dd'))
    .withColumn('ctr',
        spark_round(col('clicks').cast('double') / col('impressions').cast('double') * 100, 3)
    )
    .withColumn('ctr_band',
        when(col('ctr') >= 3.0, 'High (≥3%)')
        .when(col('ctr') >= 1.5, 'Medium (1.5–3%)')
        .otherwise('Low (<1.5%)')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
df_ads.write.mode('overwrite').format('delta').saveAsTable('silver_ad_impressions')
print(f'Silver ad impressions: {df_ads.count()} rows')

In [ ]:
print('Silver transformation complete.')
for tbl in ['silver_content_catalog', 'silver_subscribers', 'silver_viewing_history', 'silver_ad_impressions']:
    n = spark.read.format('delta').table(tbl).count()
    print(f'  {tbl:<30} {n:>8,} rows')